# Сравнение MLP и MGS-KAN по фолдам и seed

Выбери GPU и режим в первой ячейке. `smoke` проверяет обе модели на одном фолде за две эпохи. `full` запускает пять фолдов и три пары seed за 50 эпох. Обучения идут последовательно.

В окружении ядра установи проект из корня репозитория командой `python -m pip install -e '.[tracking]' pandas`. Укажи `MLFLOW_TRACKING_URI` в корневом `.env` или окружении ядра. Для Concrete нужен файл `data/concrete_compressive_strength.csv`. Чтобы разрешить его первую загрузку, выставь `DOWNLOAD = True`.

In [ ]:
MODE = "smoke"      # "smoke" или "full"
DEVICE = "cuda:0"   # Индекс среди GPU, видимых ядру. Для локальной проверки — "cpu".
DATASET = "concrete"
VERSION = "v1"      # Измени для новой серии после изменения протокола.
DOWNLOAD = False

## План серии

В каждой паре первый seed задаёт начальные веса, второй — порядок батчей. Проверь размер серии и лимиты перед запуском следующего раздела.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from config import ModelSettings, MonitorSettings, ResourceSettings, TrackingSettings, TrainingSettings
from experiments import ExperimentRunner

cwd = Path.cwd().resolve()
for project_root in (cwd, *cwd.parents):
    if (project_root / "src/experiments/runner.py").is_file():
        break
else:
    raise RuntimeError("Открой ноутбук из рабочей копии mgs-kan.")

env_file = project_root / ".env"
resources = ResourceSettings(_env_file=env_file)
models = ["mlp", "mgs_kan"]

if MODE == "smoke":
    folds = [0]
    seeds = [(11, 101)]
    epochs = 2
    max_runs = 2
elif MODE == "full":
    folds = list(range(5))
    seeds = [(11, 101), (22, 202), (33, 303)]
    epochs = 50
    max_runs = 30
else:
    raise ValueError("Выбери MODE = 'smoke' или 'full'.")

series_id = f"{DATASET}-baselines-{MODE}-{VERSION}"
planned = len(models) * len(folds) * len(seeds)
print(f"{series_id} | {DEVICE} | {planned} обучений по {epochs} эпох")
print(f"Модели: {models} | Фолды: {folds} | Пары seed: {seeds}")
print(f"Потоков CPU на обучение: {resources.cpu_threads}")
if DEVICE.startswith("cuda:"):
    print(f"Лимит памяти PyTorch на выбранной GPU: {resources.gpu_memory_fraction:.0%}")

## Запуск и возобновление

После перезапуска ядра выполни ячейки с прежними параметрами. В том же эксперименте MLflow раннер найдёт завершённые комбинации по `series_id` и продолжит с оставшихся. Для проверки повтори `smoke` и сравни `run_id` в таблице.

In [ ]:
runner = ExperimentRunner(
    training=TrainingSettings(epochs=epochs, _env_file=env_file),
    model_settings=ModelSettings(_env_file=env_file),
    resources=resources,
    tracking=TrackingSettings(_env_file=env_file),
    monitoring=MonitorSettings(_env_file=env_file),
)
summary = runner.run(
    series_id=series_id,
    dataset=DATASET,
    root=project_root / "data",
    models=models,
    folds=folds,
    seeds=seeds,
    device=DEVICE,
    download=DOWNLOAD,
    max_runs=max_runs,
)

## Результаты по обучениям

RMSE и R² рассчитаны на тестовом фолде с весами лучшей эпохи. Пик памяти показывает выделения PyTorch в MiB и доступен при запуске на CUDA.

In [ ]:
from dataclasses import asdict

rows = []
for result in summary.results:
    metrics = runner.tracker.client.get_run(result.run_id).data.metrics
    peak_bytes = metrics.get("cuda_peak_allocated_bytes")
    rows.append({
        **asdict(result.trial),
        "best_epoch": result.best_epoch,
        "test_rmse": result.test_rmse,
        "test_r2": result.test_r2,
        "duration_s": metrics.get("duration_seconds"),
        "peak_gpu_mib": peak_bytes / 2**20 if peak_bytes is not None else None,
        "run_id": result.run_id,
    })

display(pd.DataFrame(rows))

## Сводка по моделям

Сначала метрики усредняются по seed внутри каждого фолда, затем — по фолдам с равным весом. Столбцы `completed` и `planned` показывают покрытие серии.

In [ ]:
by_model = pd.DataFrame.from_dict(summary.by_model, orient="index")
display(by_model.rename_axis("model"))
print(f"Осталось обучений: {len(summary.pending)}")
if summary.pending:
    display(pd.DataFrame(asdict(trial) for trial in summary.pending))